# Train Match Edge AI V5

This notebook trains the Match Edge AI V5 calibrated specialist ensemble.

The final deployment model uses all available historical matches before the deployment cutoff, including 2024, 2025, and early 2026 data when available.

## Model target

The model predicts the match result from Team A's perspective.

- `0` = Team A loss
- `1` = Draw
- `2` = Team A win

In [9]:
# Import libraries and set project paths.

from pathlib import Path
import math
import pickle
import json
import warnings

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, PoissonRegressor
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, log_loss

warnings.filterwarnings("ignore")

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except Exception:
    XGBOOST_AVAILABLE = False

try:
    from scipy.optimize import minimize
    SCIPY_AVAILABLE = True
except Exception:
    SCIPY_AVAILABLE = False

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_DIR = ROOT / "data" / "processed"
MODEL_DIR = ROOT / "models"
REPORT_DIR = ROOT / "reports"

MODEL_DIR.mkdir(exist_ok=True)
REPORT_DIR.mkdir(exist_ok=True)

print("Project root:", ROOT)
print("Data folder:", DATA_DIR)
print("Model folder:", MODEL_DIR)
print("Report folder:", REPORT_DIR)
print("XGBoost available:", XGBOOST_AVAILABLE)
print("SciPy available:", SCIPY_AVAILABLE)

Project root: c:\Users\tevin\OneDrive\Desktop\wc_match_predictions
Data folder: c:\Users\tevin\OneDrive\Desktop\wc_match_predictions\data\processed
Model folder: c:\Users\tevin\OneDrive\Desktop\wc_match_predictions\models
Report folder: c:\Users\tevin\OneDrive\Desktop\wc_match_predictions\reports
XGBoost available: True
SciPy available: True


## Load training data

The notebook expects the processed training file at:

`data/processed/match_training_data.csv`

In [10]:
# Load the processed historical match training data.

training_path = DATA_DIR / "match_training_data.csv"

if not training_path.exists():
    raise FileNotFoundError(f"Training file not found: {training_path}")

training = pd.read_csv(training_path)

training["match_date_dt"] = pd.to_datetime(training["match_date"])
training["target_result"] = training["target_result"].astype(int)

print("Training rows:", len(training))
print("Date range:", training["match_date_dt"].min(), "to", training["match_date_dt"].max())
print("Available columns:")
print(training.columns.tolist())

# Preview only columns that actually exist in the training file.
preview_columns = [
    "match_date",
    "team_a_name",
    "team_b_name",
    "team_a",
    "team_b",
    "home_team",
    "away_team",
    "team_a_score",
    "team_b_score",
    "home_score",
    "away_score",
    "target_result",
]

preview_columns = [
    column
    for column in preview_columns
    if column in training.columns
]

training[preview_columns].head()

Training rows: 5490
Date range: 2014-01-01 00:00:00 to 2026-06-10 00:00:00
Available columns:
['match_id', 'match_date', 'team_a_raw_name', 'team_b_raw_name', 'team_a_id', 'team_b_id', 'team_a_is_wc2026', 'team_b_is_wc2026', 'tournament', 'neutral_flag', 'importance', 'team_a_pre_elo', 'team_b_pre_elo', 'pre_elo_difference', 'team_a_host_advantage', 'team_b_host_advantage', 'team_a_score', 'team_b_score', 'target_result', 'team_a_matches_last_5', 'team_a_wins_last_5', 'team_a_points_avg_last_5', 'team_a_goal_diff_avg_last_5', 'team_a_matches_last_10', 'team_a_wins_last_10', 'team_a_points_avg_last_10', 'team_a_goals_for_avg_last_10', 'team_a_goals_against_avg_last_10', 'team_a_goal_diff_avg_last_10', 'team_a_avg_opponent_elo_last_10', 'team_a_days_since_last_match', 'team_b_matches_last_5', 'team_b_wins_last_5', 'team_b_points_avg_last_5', 'team_b_goal_diff_avg_last_5', 'team_b_matches_last_10', 'team_b_wins_last_10', 'team_b_points_avg_last_10', 'team_b_goals_for_avg_last_10', 'team_b

,match_date,team_a_score,team_b_score,target_result
0,2014-01-01,1,2,0
1,2014-01-04,0,1,0
2,2014-01-04,0,1,0
3,2014-01-04,3,0,2
4,2014-01-07,2,0,2


## Date split strategy

The split is time-based to avoid future data leakage.

- Base training data fits the specialist models.
- Blend training data learns the best ensemble weights and calibration.
- Test data evaluates recent performance.
- Deployment training data fits the final saved model.

The deployment model uses all matches before `DEPLOYMENT_CUTOFF`.

In [11]:
# Create a time-based split.
# The final deployment model uses 2024, 2025, and early 2026 data when available.

BASE_TRAIN_END = "2024-01-01"
BLEND_TRAIN_START = "2024-01-01"
TEST_START = "2026-01-01"

# Pre-tournament cutoff.
# Update this date later when refreshing the model with newer results.
DEPLOYMENT_CUTOFF = "2026-06-11"

base_train_df = training[training["match_date_dt"] < BASE_TRAIN_END].copy()

blend_train_df = training[
    (training["match_date_dt"] >= BLEND_TRAIN_START)
    & (training["match_date_dt"] < TEST_START)
].copy()

test_df = training[
    (training["match_date_dt"] >= TEST_START)
    & (training["match_date_dt"] < DEPLOYMENT_CUTOFF)
].copy()

deploy_train_df = training[training["match_date_dt"] < DEPLOYMENT_CUTOFF].copy()

print("Initial split")
print("Base train rows:", len(base_train_df))
print("Blend train rows:", len(blend_train_df))
print("Test rows:", len(test_df))
print("Deploy train rows:", len(deploy_train_df))

Initial split
Base train rows: 4295
Blend train rows: 1042
Test rows: 153
Deploy train rows: 5490


In [12]:
# Use fallback split windows if the newest test window is too small.
# This keeps the notebook stable across different data refreshes.

if len(test_df) < 50:
    print("2026 test window is small. Using 2025 and early 2026 as the test window.")

    BASE_TRAIN_END = "2023-01-01"
    BLEND_TRAIN_START = "2023-01-01"
    TEST_START = "2025-01-01"

    base_train_df = training[training["match_date_dt"] < BASE_TRAIN_END].copy()

    blend_train_df = training[
        (training["match_date_dt"] >= BLEND_TRAIN_START)
        & (training["match_date_dt"] < TEST_START)
    ].copy()

    test_df = training[
        (training["match_date_dt"] >= TEST_START)
        & (training["match_date_dt"] < DEPLOYMENT_CUTOFF)
    ].copy()

    deploy_train_df = training[training["match_date_dt"] < DEPLOYMENT_CUTOFF].copy()

if len(test_df) < 50:
    print("2025+ test window is still small. Using 2024 and newer as the test window.")

    BASE_TRAIN_END = "2022-01-01"
    BLEND_TRAIN_START = "2022-01-01"
    TEST_START = "2024-01-01"

    base_train_df = training[training["match_date_dt"] < BASE_TRAIN_END].copy()

    blend_train_df = training[
        (training["match_date_dt"] >= BLEND_TRAIN_START)
        & (training["match_date_dt"] < TEST_START)
    ].copy()

    test_df = training[
        (training["match_date_dt"] >= TEST_START)
        & (training["match_date_dt"] < DEPLOYMENT_CUTOFF)
    ].copy()

    deploy_train_df = training[training["match_date_dt"] < DEPLOYMENT_CUTOFF].copy()

if len(base_train_df) == 0 or len(blend_train_df) == 0 or len(test_df) == 0:
    raise ValueError("One or more training windows are empty. Check the match date range.")

print("Final split")
print("Base train rows:", len(base_train_df))
print("Blend train rows:", len(blend_train_df))
print("Test rows:", len(test_df))
print("Deploy train rows:", len(deploy_train_df))
print("Base train window: before", BASE_TRAIN_END)
print("Blend train window:", BLEND_TRAIN_START, "through before", TEST_START)
print("Test window:", TEST_START, "through before", DEPLOYMENT_CUTOFF)
print("Deployment training window: all matches before", DEPLOYMENT_CUTOFF)

Final split
Base train rows: 4295
Blend train rows: 1042
Test rows: 153
Deploy train rows: 5490
Base train window: before 2024-01-01
Blend train window: 2024-01-01 through before 2026-01-01
Test window: 2026-01-01 through before 2026-06-11
Deployment training window: all matches before 2026-06-11


## Feature groups

Each specialist model uses the feature group that matches its job.

In [13]:
# Define feature groups for each specialist model.

strength_features = [
    "team_a_pre_elo",
    "team_b_pre_elo",
    "pre_elo_difference",
    "team_a_avg_opponent_elo_last_10",
    "team_b_avg_opponent_elo_last_10",
    "avg_opponent_elo_last_10_difference",
]

form_features = [
    "team_a_matches_last_5",
    "team_b_matches_last_5",
    "team_a_wins_last_5",
    "team_b_wins_last_5",
    "wins_last_5_difference",
    "team_a_points_avg_last_5",
    "team_b_points_avg_last_5",
    "points_avg_last_5_difference",
    "team_a_goal_diff_avg_last_5",
    "team_b_goal_diff_avg_last_5",
    "goal_diff_avg_last_5_difference",
]

attack_defense_features = [
    "team_a_goals_for_avg_last_10",
    "team_b_goals_for_avg_last_10",
    "goals_for_avg_last_10_difference",
    "team_a_goals_against_avg_last_10",
    "team_b_goals_against_avg_last_10",
    "goals_against_avg_last_10_difference",
    "team_a_goal_diff_avg_last_10",
    "team_b_goal_diff_avg_last_10",
    "goal_diff_avg_last_10_difference",
    "team_a_pre_elo",
    "team_b_pre_elo",
    "pre_elo_difference",
    "team_a_host_advantage",
    "team_b_host_advantage",
    "neutral_flag",
]

context_features = [
    "neutral_flag",
    "importance",
    "team_a_host_advantage",
    "team_b_host_advantage",
    "team_a_days_since_last_match",
    "team_b_days_since_last_match",
    "days_since_last_match_difference",
]

draw_features = [
    "pre_elo_difference",
    "team_a_points_avg_last_5",
    "team_b_points_avg_last_5",
    "points_avg_last_5_difference",
    "team_a_goal_diff_avg_last_10",
    "team_b_goal_diff_avg_last_10",
    "goal_diff_avg_last_10_difference",
    "team_a_goals_for_avg_last_10",
    "team_b_goals_for_avg_last_10",
    "team_a_goals_against_avg_last_10",
    "team_b_goals_against_avg_last_10",
    "neutral_flag",
    "importance",
]

In [14]:
# Define the full feature set and specialist order.

all_features = [
    "neutral_flag",
    "importance",
    "team_a_pre_elo",
    "team_b_pre_elo",
    "pre_elo_difference",
    "team_a_host_advantage",
    "team_b_host_advantage",
    "team_a_matches_last_5",
    "team_b_matches_last_5",
    "team_a_wins_last_5",
    "team_b_wins_last_5",
    "wins_last_5_difference",
    "team_a_points_avg_last_5",
    "team_b_points_avg_last_5",
    "points_avg_last_5_difference",
    "team_a_goal_diff_avg_last_5",
    "team_b_goal_diff_avg_last_5",
    "goal_diff_avg_last_5_difference",
    "team_a_goals_for_avg_last_10",
    "team_b_goals_for_avg_last_10",
    "goals_for_avg_last_10_difference",
    "team_a_goals_against_avg_last_10",
    "team_b_goals_against_avg_last_10",
    "goals_against_avg_last_10_difference",
    "team_a_goal_diff_avg_last_10",
    "team_b_goal_diff_avg_last_10",
    "goal_diff_avg_last_10_difference",
    "team_a_avg_opponent_elo_last_10",
    "team_b_avg_opponent_elo_last_10",
    "avg_opponent_elo_last_10_difference",
    "team_a_days_since_last_match",
    "team_b_days_since_last_match",
    "days_since_last_match_difference",
]

feature_groups = {
    "strength_elo_logistic": strength_features,
    "recent_form_random_forest": form_features,
    "poisson_scoreline_model": attack_defense_features,
    "context_xgboost": context_features,
    "full_xgboost": all_features,
    "draw_risk_classifier": draw_features,
}

specialist_order = [
    "strength_elo_logistic",
    "recent_form_random_forest",
    "poisson_scoreline_model",
    "context_xgboost",
    "full_xgboost",
    "draw_risk_classifier",
]

all_needed = sorted(
    set(
        strength_features
        + form_features
        + attack_defense_features
        + context_features
        + draw_features
        + all_features
    )
)

missing_features = [col for col in all_needed if col not in training.columns]

if missing_features:
    raise ValueError(f"Missing required training columns: {missing_features}")

print("Feature count:", len(all_needed))

Feature count: 33


In [15]:
# Clean numeric feature columns.

def clean_numeric_features(df, columns):
    cleaned = df.copy()

    cleaned[columns] = (
        cleaned[columns]
        .astype(float)
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0.0)
    )

    return cleaned


training = clean_numeric_features(training, all_needed)
base_train_df = clean_numeric_features(base_train_df, all_needed)
blend_train_df = clean_numeric_features(blend_train_df, all_needed)
test_df = clean_numeric_features(test_df, all_needed)
deploy_train_df = clean_numeric_features(deploy_train_df, all_needed)

print("Numeric features cleaned.")

Numeric features cleaned.


## Model helper functions

These functions build models and convert each specialist output into the same result format.

In [16]:
# Build XGBoost models when available.
# Fall back to scikit-learn models if XGBoost is unavailable.

def build_xgb_multiclass():
    if XGBOOST_AVAILABLE:
        return XGBClassifier(
            objective="multi:softprob",
            num_class=3,
            n_estimators=90,
            max_depth=3,
            learning_rate=0.055,
            subsample=0.9,
            colsample_bytree=0.9,
            eval_metric="mlogloss",
            random_state=42,
            n_jobs=1,
        )

    return HistGradientBoostingClassifier(
        max_iter=130,
        learning_rate=0.05,
        max_leaf_nodes=20,
        random_state=42,
    )


def build_xgb_binary():
    if XGBOOST_AVAILABLE:
        return XGBClassifier(
            objective="binary:logistic",
            n_estimators=80,
            max_depth=2,
            learning_rate=0.05,
            subsample=0.9,
            colsample_bytree=0.9,
            eval_metric="logloss",
            random_state=42,
            n_jobs=1,
        )

    return LogisticRegression(max_iter=700)

In [17]:
# Helper functions for probabilities and Poisson score modeling.

def predict_proba_aligned(model, features, classes=(0, 1, 2)):
    probs = model.predict_proba(features)
    model_classes = list(model.classes_)

    output = np.zeros((len(features), len(classes)))

    for idx, cls in enumerate(model_classes):
        cls = int(cls)

        if cls in classes:
            output[:, classes.index(cls)] = probs[:, idx]

    return output


def poisson_pmf(k, lam):
    lam = np.clip(lam, 0.05, 6.0)
    return np.exp(-lam) * np.power(lam, k) / math.factorial(k)


def poisson_1x2_from_lambdas(lambda_a, lambda_b, max_goals=7):
    rows = []

    for la, lb in zip(np.asarray(lambda_a), np.asarray(lambda_b)):
        a_probs = np.array([poisson_pmf(g, la) for g in range(max_goals + 1)])
        b_probs = np.array([poisson_pmf(g, lb) for g in range(max_goals + 1)])

        matrix = np.outer(a_probs, b_probs)

        # Class order: [Team B win, Draw, Team A win]
        a_win = float(np.tril(matrix, -1).sum())
        draw = float(np.trace(matrix))
        b_win = float(np.triu(matrix, 1).sum())

        total = max(a_win + draw + b_win, 1e-9)

        rows.append([b_win / total, draw / total, a_win / total])

    return np.array(rows)


def draw_to_1x2_probs(draw_prob, anchor_probs):
    rows = []

    for draw_value, anchor in zip(draw_prob, anchor_probs):
        draw_value = float(np.clip(draw_value, 0.05, 0.70))

        non_draw = 1.0 - draw_value

        b_weight = max(float(anchor[0]), 1e-6)
        a_weight = max(float(anchor[2]), 1e-6)

        total = b_weight + a_weight

        rows.append(
            [
                non_draw * b_weight / total,
                draw_value,
                non_draw * a_weight / total,
            ]
        )

    return np.array(rows)

## Specialist model training

Each model learns a different match signal.

In [18]:
# Train each specialist model.

def fit_specialists(df):
    models = {
        "strength_elo_logistic": Pipeline(
            [
                ("scaler", StandardScaler()),
                ("model", LogisticRegression(max_iter=800)),
            ]
        ),
        "recent_form_random_forest": RandomForestClassifier(
            n_estimators=150,
            max_depth=7,
            min_samples_leaf=10,
            random_state=42,
            n_jobs=-1,
        ),
        "poisson_team_a_goals": Pipeline(
            [
                ("scaler", StandardScaler()),
                ("model", PoissonRegressor(alpha=0.01, max_iter=800)),
            ]
        ),
        "poisson_team_b_goals": Pipeline(
            [
                ("scaler", StandardScaler()),
                ("model", PoissonRegressor(alpha=0.01, max_iter=800)),
            ]
        ),
        "context_xgboost": build_xgb_multiclass(),
        "full_xgboost": build_xgb_multiclass(),
        "draw_risk_classifier": build_xgb_binary(),
    }

    models["strength_elo_logistic"].fit(df[strength_features], df["target_result"])

    models["recent_form_random_forest"].fit(df[form_features], df["target_result"])

    models["poisson_team_a_goals"].fit(
        df[attack_defense_features],
        df["team_a_score"].astype(float),
    )

    models["poisson_team_b_goals"].fit(
        df[attack_defense_features],
        df["team_b_score"].astype(float),
    )

    models["context_xgboost"].fit(df[context_features], df["target_result"])

    models["full_xgboost"].fit(df[all_features], df["target_result"])

    draw_target = (df["target_result"] == 1).astype(int)

    models["draw_risk_classifier"].fit(df[draw_features], draw_target)

    return models

In [19]:
# Generate probability outputs from each specialist model.

def specialist_probability_map(models, df):
    strength_probs = predict_proba_aligned(
        models["strength_elo_logistic"],
        df[strength_features],
    )

    form_probs = predict_proba_aligned(
        models["recent_form_random_forest"],
        df[form_features],
    )

    lambda_a = np.clip(
        models["poisson_team_a_goals"].predict(df[attack_defense_features]),
        0.05,
        6.0,
    )

    lambda_b = np.clip(
        models["poisson_team_b_goals"].predict(df[attack_defense_features]),
        0.05,
        6.0,
    )

    poisson_probs = poisson_1x2_from_lambdas(lambda_a, lambda_b)

    context_probs = predict_proba_aligned(
        models["context_xgboost"],
        df[context_features],
    )

    full_probs = predict_proba_aligned(
        models["full_xgboost"],
        df[all_features],
    )

    draw_raw = models["draw_risk_classifier"].predict_proba(df[draw_features])

    draw_prob = (
        draw_raw[:, 1]
        if draw_raw.shape[1] > 1
        else np.repeat(0.25, len(df))
    )

    draw_probs = draw_to_1x2_probs(draw_prob, full_probs)

    return {
        "strength_elo_logistic": strength_probs,
        "recent_form_random_forest": form_probs,
        "poisson_scoreline_model": poisson_probs,
        "context_xgboost": context_probs,
        "full_xgboost": full_probs,
        "draw_risk_classifier": draw_probs,
    }


def blended_probs(prob_map, weights):
    probs = np.zeros_like(next(iter(prob_map.values())))

    for name, weight in zip(specialist_order, weights):
        probs += weight * prob_map[name]

    return probs

## Blend optimization and calibration

The blend optimizer learns how much each specialist should contribute.

Temperature calibration improves probability quality.

In [20]:
# Optimize blend weights and calibrate probabilities.

def temperature_scale_probs(probs, temperature):
    probs = np.clip(probs, 1e-8, 1.0)

    logits = np.log(probs) / float(temperature)
    logits = logits - logits.max(axis=1, keepdims=True)

    exp_logits = np.exp(logits)

    return exp_logits / exp_logits.sum(axis=1, keepdims=True)


def fit_temperature(probs, target):
    best_temperature = 1.0
    best_loss = log_loss(target, probs, labels=[0, 1, 2])

    for temperature in np.linspace(0.65, 2.2, 64):
        scaled = temperature_scale_probs(probs, temperature)
        loss = log_loss(target, scaled, labels=[0, 1, 2])

        if loss < best_loss:
            best_temperature = float(temperature)
            best_loss = float(loss)

    return best_temperature, best_loss


def optimize_blend_weights(prob_list, target):
    n_models = len(prob_list)

    prob_array = np.stack(prob_list, axis=0)

    def make_weighted_probs(weights):
        weights = np.asarray(weights)
        weights = weights / max(weights.sum(), 1e-12)

        return np.tensordot(weights, prob_array, axes=(0, 0))

    def objective(weights):
        probs = make_weighted_probs(weights)

        return log_loss(target, probs, labels=[0, 1, 2])

    start_weights = np.ones(n_models) / n_models

    if SCIPY_AVAILABLE:
        constraints = {
            "type": "eq",
            "fun": lambda weights: np.sum(weights) - 1,
        }

        bounds = [(0, 1) for _ in range(n_models)]

        result = minimize(
            objective,
            start_weights,
            method="SLSQP",
            bounds=bounds,
            constraints=constraints,
            options={"maxiter": 500},
        )

        if result.success:
            weights = np.clip(result.x, 0, 1)
            weights = weights / weights.sum()

            return weights, float(objective(weights)), "scipy_slsqp"

    rng = np.random.default_rng(42)

    best_weights = start_weights.copy()
    best_loss = objective(best_weights)

    for _ in range(5000):
        weights = rng.dirichlet(np.ones(n_models))
        loss = objective(weights)

        if loss < best_loss:
            best_weights = weights
            best_loss = loss

    return best_weights, float(best_loss), "dirichlet_random_search"

## Train and evaluate

Specialist models train on the base window.

Blend weights and calibration train on the blend window.

Evaluation happens on the most recent holdout window.

In [21]:
# Train specialist models and learn blend weights.

eval_specialists = fit_specialists(base_train_df)

blend_train_prob_map = specialist_probability_map(
    eval_specialists,
    blend_train_df,
)

prob_list = [
    blend_train_prob_map[name]
    for name in specialist_order
]

weights, blend_train_loss, weight_method = optimize_blend_weights(
    prob_list,
    blend_train_df["target_result"].values,
)

raw_blend_train_probs = blended_probs(
    blend_train_prob_map,
    weights,
)

temperature, calibrated_train_loss = fit_temperature(
    raw_blend_train_probs,
    blend_train_df["target_result"].values,
)

weights_df = pd.DataFrame(
    {
        "specialist": specialist_order,
        "optimized_weight": weights,
    }
)

print("Weight method:", weight_method)
print("Blend train log loss:", round(blend_train_loss, 4))
print("Calibration temperature:", round(temperature, 4))
print("Calibrated train log loss:", round(calibrated_train_loss, 4))

weights_df

Weight method: scipy_slsqp
Blend train log loss: 0.8569
Calibration temperature: 0.9452
Calibrated train log loss: 0.8562


,specialist,optimized_weight
0,strength_elo_logistic,0.404705
1,recent_form_random_forest,0.111795
2,poisson_scoreline_model,0.085487
3,context_xgboost,0.059879
4,full_xgboost,0.162845
5,draw_risk_classifier,0.175289


In [22]:
# Evaluate the model on recent holdout data.

test_prob_map = specialist_probability_map(
    eval_specialists,
    test_df,
)

raw_blend_test_probs = blended_probs(
    test_prob_map,
    weights,
)

calibrated_test_probs = temperature_scale_probs(
    raw_blend_test_probs,
    temperature,
)

metrics = []

scenario_names = {
    "strength_elo_logistic": "overall team strength",
    "recent_form_random_forest": "recent form",
    "poisson_scoreline_model": "expected score / scoreline",
    "context_xgboost": "match context",
    "full_xgboost": "full nonlinear 1X2",
    "draw_risk_classifier": "draw risk",
}

for name in specialist_order:
    probs = test_prob_map[name]
    preds = probs.argmax(axis=1)

    metrics.append(
        {
            "model": name,
            "scenario": scenario_names[name],
            "role": "specialist",
            "accuracy": round(
                float(accuracy_score(test_df["target_result"], preds)),
                4,
            ),
            "log_loss": round(
                float(
                    log_loss(
                        test_df["target_result"],
                        probs,
                        labels=[0, 1, 2],
                    )
                ),
                4,
            ),
            "base_train_rows": len(base_train_df),
            "blend_train_rows": len(blend_train_df),
            "test_rows": len(test_df),
        }
    )

raw_preds = raw_blend_test_probs.argmax(axis=1)

calibrated_preds = calibrated_test_probs.argmax(axis=1)

metrics.append(
    {
        "model": "optimized_weighted_blend_raw",
        "scenario": "optimized blend before calibration",
        "role": "final",
        "accuracy": round(
            float(accuracy_score(test_df["target_result"], raw_preds)),
            4,
        ),
        "log_loss": round(
            float(
                log_loss(
                    test_df["target_result"],
                    raw_blend_test_probs,
                    labels=[0, 1, 2],
                )
            ),
            4,
        ),
        "base_train_rows": len(base_train_df),
        "blend_train_rows": len(blend_train_df),
        "test_rows": len(test_df),
    }
)

metrics.append(
    {
        "model": "optimized_weighted_blend_calibrated",
        "scenario": "optimized blend plus temperature calibration",
        "role": "final",
        "accuracy": round(
            float(accuracy_score(test_df["target_result"], calibrated_preds)),
            4,
        ),
        "log_loss": round(
            float(
                log_loss(
                    test_df["target_result"],
                    calibrated_test_probs,
                    labels=[0, 1, 2],
                )
            ),
            4,
        ),
        "base_train_rows": len(base_train_df),
        "blend_train_rows": len(blend_train_df),
        "test_rows": len(test_df),
    }
)

metrics_df = pd.DataFrame(metrics).sort_values(
    ["role", "log_loss"]
)

metrics_df

,model,scenario,role,accuracy,log_loss,base_train_rows,blend_train_rows,test_rows
7,optimized_weighted_blend_calibrated,optimized blend plus temperature calibration,final,0.6013,0.8712,4295,1042,153
6,optimized_weighted_blend_raw,optimized blend before calibration,final,0.6013,0.8740,4295,1042,153
0,strength_elo_logistic,overall team strength,specialist,0.6144,0.8688,4295,1042,153
4,full_xgboost,full nonlinear 1X2,specialist,0.5948,0.8724,4295,1042,153
5,draw_risk_classifier,draw risk,specialist,0.5882,0.8788,4295,1042,153
2,poisson_scoreline_model,expected score / scoreline,specialist,0.5817,0.8819,4295,1042,153
1,recent_form_random_forest,recent form,specialist,0.5359,0.9410,4295,1042,153
3,context_xgboost,match context,specialist,0.5033,1.0012,4295,1042,153


## Save deployment model

The final deployment model trains on all matches before the deployment cutoff.

Streamlit loads the saved `.pkl` file from the `models` folder.

In [23]:
# Train the final deployment model.
# This model uses all matches before the deployment cutoff.
# This is the model Streamlit loads.

deploy_specialists = fit_specialists(deploy_train_df)

metadata = {
    "model_version": "match_edge_ai_v5_calibrated_specialist_ensemble_recent_training",
    "model_type": "Calibrated specialist ensemble with optimized weighted blend",
    "xgboost_available": bool(XGBOOST_AVAILABLE),
    "scipy_available": bool(SCIPY_AVAILABLE),
    "weight_optimizer": weight_method,
    "optimized_weights": {
        name: float(weight)
        for name, weight in zip(specialist_order, weights)
    },
    "temperature": float(temperature),
    "target_mapping": {
        "0": "Team A loss",
        "1": "Draw",
        "2": "Team A win",
    },
    "base_train_window": f"before {BASE_TRAIN_END}",
    "blend_train_window": f"{BLEND_TRAIN_START} through before {TEST_START}",
    "test_window": f"{TEST_START} through before {DEPLOYMENT_CUTOFF}",
    "deployment_training_window": f"all matches before {DEPLOYMENT_CUTOFF}",
    "prediction_scope": "Only the 48 World Cup 2026 teams are accepted.",
}

bundle = {
    "model_version": metadata["model_version"],
    "specialists": deploy_specialists,
    "feature_groups": feature_groups,
    "specialist_order": specialist_order,
    "weights": weights,
    "temperature": temperature,
    "metadata": metadata,
}

model_path = MODEL_DIR / "match_edge_ai_v5_calibrated_specialist_ensemble.pkl"

with open(model_path, "wb") as file:
    pickle.dump(bundle, file)

metrics_df.to_csv(
    REPORT_DIR / "model_metrics.csv",
    index=False,
)

weights_df.to_csv(
    REPORT_DIR / "optimized_blend_weights.csv",
    index=False,
)

(REPORT_DIR / "model_metadata.json").write_text(
    json.dumps(metadata, indent=2)
)

print("Saved model:", model_path)
print("Saved metrics:", REPORT_DIR / "model_metrics.csv")
print("Saved weights:", REPORT_DIR / "optimized_blend_weights.csv")

Saved model: c:\Users\tevin\OneDrive\Desktop\wc_match_predictions\models\match_edge_ai_v5_calibrated_specialist_ensemble.pkl
Saved metrics: c:\Users\tevin\OneDrive\Desktop\wc_match_predictions\reports\model_metrics.csv
Saved weights: c:\Users\tevin\OneDrive\Desktop\wc_match_predictions\reports\optimized_blend_weights.csv


## Training summary

The final output includes the trained model, metrics, blend weights, and model metadata.

In [24]:
# Final training summary.

print("Model training complete.")
print("Deployment cutoff:", DEPLOYMENT_CUTOFF)
print("Rows used for final deployed model:", len(deploy_train_df))
print("Saved model:", model_path)

metrics_df

Model training complete.
Deployment cutoff: 2026-06-11
Rows used for final deployed model: 5490
Saved model: c:\Users\tevin\OneDrive\Desktop\wc_match_predictions\models\match_edge_ai_v5_calibrated_specialist_ensemble.pkl


,model,scenario,role,accuracy,log_loss,base_train_rows,blend_train_rows,test_rows
7,optimized_weighted_blend_calibrated,optimized blend plus temperature calibration,final,0.6013,0.8712,4295,1042,153
6,optimized_weighted_blend_raw,optimized blend before calibration,final,0.6013,0.8740,4295,1042,153
0,strength_elo_logistic,overall team strength,specialist,0.6144,0.8688,4295,1042,153
4,full_xgboost,full nonlinear 1X2,specialist,0.5948,0.8724,4295,1042,153
5,draw_risk_classifier,draw risk,specialist,0.5882,0.8788,4295,1042,153
2,poisson_scoreline_model,expected score / scoreline,specialist,0.5817,0.8819,4295,1042,153
1,recent_form_random_forest,recent form,specialist,0.5359,0.9410,4295,1042,153
3,context_xgboost,match context,specialist,0.5033,1.0012,4295,1042,153
